In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/aspideraman/krishi-darshan-progres/krishi_darshan_transcripts.csv


In [2]:

# %% [markdown]
# # 🌾 Krishi Darshan — YouTube Transcript Extractor
# **Pipeline:** Playlist → Audio → Whisper Transcription → Structured CSV
#
# - Source: Doordarshan National (74 episodes)
# - Model: openai/whisper-large-v3 (HuggingFace)
# - Checkpoint system: resumes from last completed video if session crashes
# - Output: master CSV with episode metadata + full transcript
 
# %% [markdown]

In [3]:
# ## Cell 1 — Install Dependencies (run once)
 
# %%
!pip install yt-dlp transformers torch torchaudio pandas -q
!apt-get install -y ffmpeg -q   # required by yt-dlp for audio conversion
 
# %% [markdown]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 71.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 97.6 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numb

In [4]:
# ## Cell 2 — Configuration
 
# %%
import os
 
PLAYLIST_URL = "https://www.youtube.com/playlist?list=PLUiMfS6qzIMxC3SpknSSQ85dhfyPEz-rb"
 
# HuggingFace token (read-only is fine)
# On Kaggle: from kaggle_secrets import UserSecretsClient
#            HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
HF_TOKEN = os.environ.get("HF_TOKEN", "hf_REPLACE_WITH_YOUR_TOKEN")
 
# Whisper model — use "openai/whisper-medium" if large-v3 runs out of VRAM
WHISPER_MODEL = "openai/whisper-medium"
 
# Output paths
OUTPUT_CSV      = "/kaggle/working/krishi_darshan_transcripts.csv"
CHECKPOINT_FILE = "/kaggle/working/checkpoint.txt"   # stores last completed video index
AUDIO_TMP       = "/kaggle/working/tmp_audio.wav"    # reused for each video
 
# Max videos to process in one session (Kaggle 12hr limit — set lower if cautious)
MAX_VIDEOS = 74   # set to e.g. 20 for a test run
 
print("✅ Config set")
print(f"   Model   : {WHISPER_MODEL}")
print(f"   Output  : {OUTPUT_CSV}")
print(f"   Max vids: {MAX_VIDEOS}")
 
# %% [markdown]

✅ Config set
   Model   : openai/whisper-medium
   Output  : /kaggle/working/krishi_darshan_transcripts.csv
   Max vids: 74


In [5]:
# ## Cell 3 — Load Whisper Model
 
# %%
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
 
print("⏳ Loading Whisper model...")
 
device     = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
 
print(f"   Device: {device}")
 
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    WHISPER_MODEL,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True,
    use_safetensors=True,
    #token=HF_TOKEN,
)
model.to(device)
 
processor = AutoProcessor.from_pretrained(WHISPER_MODEL)
 
whisper_pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
    return_timestamps=True,          # get word-level timestamps
    generate_kwargs={"language": "hi", "task": "transcribe"},  # Hindi transcription
)
 
print("✅ Whisper loaded successfully")
 
# %% [markdown]

⏳ Loading Whisper model...
   Device: cuda


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.06G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

...

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
Passing `generation_config` together with generation-related arguments=({'return_timestamps'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✅ Whisper loaded successfully


In [6]:
# ## Cell 4 — Fetch Playlist Metadata
 
# %%
import json, subprocess
 
print("⏳ Fetching playlist metadata...")
 
# Pull metadata for all videos without downloading any audio
result = subprocess.run(
    [
        "yt-dlp",
        "--flat-playlist",
        "--dump-single-json",
        "--no-warnings",
        PLAYLIST_URL
    ],
    capture_output=True, text=True
)
 
playlist_data = json.loads(result.stdout)
entries = playlist_data.get("entries", [])
 
print(f"✅ Found {len(entries)} videos in playlist")
print(f"   Playlist title: {playlist_data.get('title', 'N/A')}")
print(f"\n   First 3 episodes:")
for e in entries[:3]:
    print(f"   [{e.get('id')}] {e.get('title', 'N/A')}")
 
# %% [markdown]

⏳ Fetching playlist metadata...
✅ Found 74 videos in playlist
   Playlist title: Krishi Darshan

   First 3 episodes:
   [SxgaGXcQ8ZY] Krishi Darshan  एकिकृत कृषि प्रणाली
   [JLwhLr4ZVd0] Krishi Darshan   Bhagwani ke vikas ka harayana me prayas
   [ET8rwerJTg0] KRISHI DARSHAN GAON SAMRIDH BHART SAMRIDH


In [7]:
# ## Cell 5 — Checkpoint Helper Functions
 
# %%
import csv, time
 
def load_checkpoint():
    """Returns index of last successfully processed video (0 if fresh start)."""
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r") as f:
            val = f.read().strip()
            return int(val) if val.isdigit() else 0
    return 0
 
def save_checkpoint(index):
    with open(CHECKPOINT_FILE, "w") as f:
        f.write(str(index))
 
def append_to_csv(row: dict):
    """Appends one episode row to the master CSV. Creates file with headers if needed."""
    file_exists = os.path.exists(OUTPUT_CSV)
    with open(OUTPUT_CSV, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "episode_id", "title", "upload_date", "duration_seconds",
            "video_url", "language_detected", "transcript", "word_count", "status"
        ])
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)
 
def cleanup_audio():
    if os.path.exists(AUDIO_TMP):
        os.remove(AUDIO_TMP)
 
print("✅ Checkpoint helpers ready")
start_index = load_checkpoint()
print(f"   Resuming from video index: {start_index}")
 
# %% [markdown]

✅ Checkpoint helpers ready
   Resuming from video index: 0


In [8]:
# ## Cell 6 — Download Audio Helper
 
# %%
def download_audio(video_url: str, output_path: str) -> bool:
    """
    Downloads audio from a YouTube video and converts to WAV.
    Returns True if successful, False if failed.
    """
    cmd = [
        "yt-dlp",
        "-x",                          # extract audio only
        "--audio-format", "wav",
        "--audio-quality", "0",        # best quality
        "--no-playlist",
        "--no-warnings",
        "-o", output_path.replace(".wav", ".%(ext)s"),
        video_url
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
 
    # yt-dlp sometimes outputs mp3/m4a — check for any audio file
    base = output_path.replace(".wav", "")
    for ext in ["wav", "mp3", "m4a", "opus", "webm"]:
        candidate = f"{base}.{ext}"
        if os.path.exists(candidate):
            if candidate != output_path:
                os.rename(candidate, output_path)
            return True
 
    print(f"      ⚠️  Download failed: {result.stderr[:200]}")
    return False
 
print("✅ Download helper ready")
 
# %% [markdown]

✅ Download helper ready


In [9]:
import shutil, os

# Restore CSV from uploaded dataset
shutil.copy(
    "/kaggle/input/datasets/aspideraman/krishi-darshan-progres/krishi_darshan_transcripts.csv",
    "/kaggle/working/krishi_darshan_transcripts.csv"
)

# Restore checkpoint from uploaded dataset if it exists
checkpoint_src = "/kaggle/input/krishi-darshan-progress/checkpoint.txt"
if os.path.exists(checkpoint_src):
    shutil.copy(checkpoint_src, "/kaggle/working/checkpoint.txt")
    print(f"✅ Restored checkpoint: {load_checkpoint()}")
else:
    # Fallback: count rows in CSV as checkpoint
    import pandas as pd
    df = pd.read_csv("/kaggle/working/krishi_darshan_transcripts.csv")
    completed = len(df[df["status"] == "success"])
    save_checkpoint(completed)
    print(f"✅ Checkpoint inferred from CSV: {completed} videos done")

✅ Checkpoint inferred from CSV: 3 videos done


In [11]:
# ## Cell 7 — Main Extraction Loop
 
# %%
import gc
 
start_index = load_checkpoint()
videos_to_process = entries[start_index:MAX_VIDEOS]
 
print(f"\n🚀 Starting extraction")
print(f"   Total videos : {len(entries)}")
print(f"   Processing   : {len(videos_to_process)} (from index {start_index})")
print("=" * 60)
 
for i, entry in enumerate(videos_to_process):
    global_index = start_index + i
    video_id  = entry.get("id", f"unknown_{global_index}")
    title     = entry.get("title", "Unknown Title")
    video_url = f"https://www.youtube.com/watch?v={video_id}"
    duration  = entry.get("duration", 0)
    upload    = entry.get("upload_date", "N/A")
 
    print(f"\n[{global_index+1:02d}/{min(MAX_VIDEOS, len(entries))}] {title[:60]}...")
 
    cleanup_audio()   # ensure clean state
 
    # ── Step 1: Download audio ────────────────────────────────
    print(f"   ⏳ Downloading audio...")
    t0 = time.time()
    success = download_audio(video_url, AUDIO_TMP)
 
    if not success:
        print(f"   ❌ Skipping — audio download failed")
        append_to_csv({
            "episode_id": video_id, "title": title, "upload_date": upload,
            "duration_seconds": duration, "video_url": video_url,
            "language_detected": "N/A", "transcript": "DOWNLOAD_FAILED",
            "word_count": 0, "status": "failed"
        })
        save_checkpoint(global_index + 1)
        continue
 
    dl_time = round(time.time() - t0)
    print(f"   ✅ Downloaded in {dl_time}s")
 
    # ── Step 2: Transcribe ────────────────────────────────────
    print(f"   ⏳ Transcribing with Whisper...")
    t1 = time.time()
    try:
        result = whisper_pipe(AUDIO_TMP)
        transcript    = result["text"].strip()
        lang_detected = result.get("language", "hi")
        word_count    = len(transcript.split())
        tx_time       = round(time.time() - t1)
        print(f"   ✅ Transcribed in {tx_time}s | {word_count} words | lang: {lang_detected}")
        status = "success"
    except Exception as e:
        print(f"   ❌ Transcription failed: {e}")
        transcript    = f"TRANSCRIPTION_ERROR: {str(e)[:100]}"
        lang_detected = "N/A"
        word_count    = 0
        status        = "failed"
 
    # ── Step 3: Save row ──────────────────────────────────────
    append_to_csv({
        "episode_id"      : video_id,
        "title"           : title,
        "upload_date"     : upload,
        "duration_seconds": duration,
        "video_url"       : video_url,
        "language_detected": lang_detected,
        "transcript"      : transcript,
        "word_count"      : word_count,
        "status"          : status
    })
 
    save_checkpoint(global_index + 1)
 
    # ── Cleanup ───────────────────────────────────────────────
    cleanup_audio()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
 
    print(f"   💾 Saved. Checkpoint: {global_index + 1}")
 
print("\n" + "=" * 60)
print("✅ Session complete")
 
# %% [markdown]


🚀 Starting extraction
   Total videos : 74
   Processing   : 67 (from index 7)

[08/74] Krishi Darshan जायद की फसलों का कीटों से बचाव...
   ⏳ Downloading audio...
   ✅ Downloaded in 6s
   ⏳ Transcribing with Whisper...


KeyboardInterrupt: 

In [ ]:
# ## Cell 8 — Preview Results
 
# %%
import pandas as pd
 
if os.path.exists(OUTPUT_CSV):
    df = pd.read_csv(OUTPUT_CSV)
    print(f"📊 Transcripts saved: {len(df)} episodes")
    print(f"   Successful : {len(df[df['status']=='success'])}")
    print(f"   Failed     : {len(df[df['status']=='failed'])}")
    print(f"   Avg words  : {df[df['status']=='success']['word_count'].mean():.0f} per episode")
    print(f"\n   Sample transcript (first 300 chars):")
    sample = df[df['status']=='success'].iloc[0] if len(df[df['status']=='success']) > 0 else None
    if sample is not None:
        print(f"   [{sample['title'][:50]}]")
        print(f"   {sample['transcript'][:300]}...")
    display(df[["episode_id","title","duration_seconds","word_count","language_detected","status"]].head(10))
else:
    print("No output file found yet — run Cell 7 first.")
 
# %% [markdown]

In [ ]:
# ---
# ## 📋 Quick Reference
#
# ### Install commands (Cell 1)
# ```bash
# !pip install yt-dlp transformers torch torchaudio pandas -q
# !apt-get install -y ffmpeg -q
# ```
#
# ### Kaggle token setup
# ```python
# from kaggle_secrets import UserSecretsClient
# import os
# os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
# ```
#
# ### If session crashes mid-run
# Just re-run Cell 7 — checkpoint.txt stores the last completed index
# and the loop resumes automatically from where it left off.
#
# ### To test on 3 videos first
# Set MAX_VIDEOS = 3 in Cell 2 before running the full 74.
#
# ### Model swap if VRAM runs out
# Change WHISPER_MODEL to "openai/whisper-medium" in Cell 2.
# Medium is ~4x faster, slightly less accurate on Hindi.
#
# ### Output CSV columns
# | Column | Description |
# |--------|-------------|
# | episode_id | YouTube video ID |
# | title | Episode title |
# | upload_date | YYYYMMDD format |
# | duration_seconds | Video length |
# | video_url | Full YouTube URL |
# | language_detected | ISO code (hi, pa, etc.) |
# | transcript | Full transcript text |
# | word_count | Number of words |
# | status | success / failed |
 

In [ ]:
import pandas as pd
df = pd.read_csv("/kaggle/working/krishi_darshan_transcripts.csv")
display(df[["title", "word_count", "language_detected", "status"]])

In [ ]:
# Run this before ending every session
from IPython.display import FileLink
display(FileLink('krishi_darshan_transcripts.csv'))
display(FileLink('checkpoint.txt'))